# DC Bike Rentals — Neural Network + XGBoost Ensemble Forecasting

**Goal:** Predict hourly bike rental counts (`total`) for a DC bikeshare system by ensembling a Neural Network and an XGBoost model, averaging their predictions for the final output.

**Dataset:** Hourly bike rental records from Washington D.C., with weather, seasonal, and temporal features. Source: [BYU-Idaho CSE 450](https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv).

**Approach:**
- Feature engineering: datetime decomposition, total rider count, high-usage event flagging
- Neural Network (TensorFlow/Keras) for capturing non-linear temporal patterns
- XGBoost Regressor for handling feature interactions and tabular structure
- Final prediction: simple average of NN and XGBoost outputs
- Evaluation: RMSE and R² on held-out test set

**Key insight:** Political/special events drive extreme spikes in casual ridership — flagging these improves both models.


## 0. Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['tensorflow', 'lets-plot']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], check=True)


## 1. Imports & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lets_plot import (
    ggplot, aes, geom_line, geom_point, geom_bar, geom_boxplot,
    ggtitle, xlab, ylab, theme_bw, LetsPlot
)
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score

LetsPlot.setup_html()
print(f"TensorFlow version: {tf.__version__}")

# ── Load data ─────────────────────────────────────────────────────────────────
bikes = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv')
print(f"Shape: {bikes.shape}")
bikes.tail()


## 2. Exploratory Data Analysis

Key questions:
1. What drives extreme spikes in casual ridership?
2. Are there clear temporal patterns (hour, weekday, season)?
3. How do weather conditions affect total rentals?


In [ ]:
bikes.describe()


In [ ]:
# ── High casual ridership days ────────────────────────────────────────────────
# Observation: political events and holidays drive extreme spikes in casual use.
high_casual = bikes[bikes['casual'] > 1000]
print(f"Records with casual > 1000: {len(high_casual)}")
print(high_casual[['dteday', 'casual', 'registered', 'weathersit', 'holiday', 'workingday']].to_string())


In [ ]:
# ── Hourly rental patterns ────────────────────────────────────────────────────
# Peak hours: 6–8am and 4–6pm on weekdays (commuter pattern)
# Weekends: flatter curve, active 8am–8pm
hourly_avg = bikes.groupby(['hr', 'workingday'])['cnt'].mean().reset_index()
print("Mean hourly rentals by workingday status:")
print(hourly_avg.pivot(index='hr', columns='workingday', values='cnt').round(1))


## 3. Feature Engineering

| Feature | Description |
|---|---|
| `datetime` | Parsed from `dteday` |
| `month` | Month of year (1–12) |
| `day` | Day of month |
| `day_of_week` | Weekday (0=Mon … 6=Sun) |
| `year` | Year (2011 or 2012) |
| `total` | `casual + registered` — model target |
| `high_event_day` | Flag for days with unusually high casual ridership (> p95) |


In [ ]:
# ── Datetime decomposition ────────────────────────────────────────────────────
df = bikes.copy()
df['datetime']    = pd.to_datetime(df['dteday'])
df['month']       = df['datetime'].dt.month
df['day']         = df['datetime'].dt.day
df['day_of_week'] = df['datetime'].dt.dayofweek
df['year']        = df['datetime'].dt.year
df = df.drop(columns=['dteday'])

# ── Target variable ───────────────────────────────────────────────────────────
df['total'] = df['casual'] + df['registered']

# ── High-event day flag ───────────────────────────────────────────────────────
# Days with extreme casual ridership are often political events / major holidays.
casual_p95 = df['casual'].quantile(0.95)
df['high_event_day'] = (df['casual'] > casual_p95).astype(int)
print(f"Casual p95 threshold: {casual_p95:.0f}")
print(f"High-event records:   {df['high_event_day'].sum()} ({df['high_event_day'].mean():.1%})")

print(f"\nFinal shape: {df.shape}")
df.head()


## 4. Train / Validation / Test Split & Scaling

In [ ]:
# ── Feature selection ────────────────────────────────────────────────────────
# Drop leaky columns: casual + registered are components of the target.
FEATURE_COLS = [
    'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday',
    'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
    'month', 'day', 'day_of_week', 'year', 'high_event_day'
]
TARGET_COL = 'total'

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

# ── Temporal-aware split ──────────────────────────────────────────────────────
# Use last 20% of data as test set to respect time ordering.
split_idx = int(len(X) * 0.8)
X_trainval, X_test = X[:split_idx], X[split_idx:]
y_trainval, y_test = y[:split_idx], y[split_idx:]

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15, random_state=42, shuffle=True
)

# ── Scaling (for NN only) ──────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,} | Val: {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}")


## 5. Neural Network Model

Architecture: 3-layer MLP with Batch Normalization and Dropout for regularization.  
EarlyStopping on validation RMSE prevents overfitting.


In [ ]:
def build_nn(input_dim: int) -> Sequential:
    """Build a regularized MLP for bike rental regression."""
    model = Sequential([
        Dense(256, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])
    return model

nn_model = build_nn(X_train_scaled.shape[1])
nn_model.summary()


In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1)
]

history = nn_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=200, batch_size=256,
    callbacks=callbacks, verbose=2
)


In [ ]:
# ── Training curve ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_xlabel('Epoch', fontweight='bold')
ax.set_ylabel('MSE Loss', fontweight='bold')
ax.set_title('Neural Network Training Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

# ── NN evaluation ──────────────────────────────────────────────────────────────
nn_val_pred  = nn_model.predict(X_val_scaled).flatten()
nn_test_pred = nn_model.predict(X_test_scaled).flatten()

print(f"NN — Val  RMSE: {root_mean_squared_error(y_val,  nn_val_pred):.2f} | R²: {r2_score(y_val,  nn_val_pred):.4f}")
print(f"NN — Test RMSE: {root_mean_squared_error(y_test, nn_test_pred):.2f} | R²: {r2_score(y_test, nn_test_pred):.4f}")


## 6. XGBoost Model

XGBoost operates directly on unscaled features — tree-based models don't require normalization.


In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.5,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=30
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)


In [ ]:
xgb_val_pred  = xgb_model.predict(X_val)
xgb_test_pred = xgb_model.predict(X_test)

print(f"XGBoost — Val  RMSE: {root_mean_squared_error(y_val,  xgb_val_pred):.2f} | R²: {r2_score(y_val,  xgb_val_pred):.4f}")
print(f"XGBoost — Test RMSE: {root_mean_squared_error(y_test, xgb_test_pred):.2f} | R²: {r2_score(y_test, xgb_test_pred):.4f}")

# ── Feature importance ─────────────────────────────────────────────────────────
fi = pd.DataFrame({'feature': FEATURE_COLS, 'importance': xgb_model.feature_importances_})
fi = fi.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi['feature'][:15][::-1], fi['importance'][:15][::-1], color='steelblue')
ax.set_xlabel('Feature Importance', fontweight='bold')
ax.set_title('Top 15 XGBoost Feature Importances', fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Ensemble — Simple Average of NN + XGBoost

**Rationale:** NN and XGBoost learn different aspects of the data — the NN captures smooth temporal patterns while XGBoost handles sharp feature interactions. Averaging reduces variance and typically outperforms either model alone.


In [ ]:
# ── Validation ensemble ───────────────────────────────────────────────────────
ensemble_val_pred  = (nn_val_pred  + xgb_val_pred)  / 2
ensemble_test_pred = (nn_test_pred + xgb_test_pred) / 2

print("=" * 55)
print(f"{'Model':<15} {'Val RMSE':>10} {'Val R²':>8} {'Test RMSE':>11} {'Test R²':>9}")
print("=" * 55)
for name, vp, tp in [
    ('NN',       nn_val_pred,       nn_test_pred),
    ('XGBoost',  xgb_val_pred,      xgb_test_pred),
    ('Ensemble', ensemble_val_pred,  ensemble_test_pred),
]:
    vr = root_mean_squared_error(y_val,  vp)
    tr = root_mean_squared_error(y_test, tp)
    vr2 = r2_score(y_val,  vp)
    tr2 = r2_score(y_test, tp)
    print(f"{name:<15} {vr:>10.2f} {vr2:>8.4f} {tr:>11.2f} {tr2:>9.4f}")
print("=" * 55)


In [ ]:
# ── Actual vs. Predicted plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = [
    ('Neural Network',  nn_test_pred),
    ('XGBoost',         xgb_test_pred),
    ('Ensemble',        ensemble_test_pred),
]
for ax, (name, pred) in zip(axes, models):
    ax.scatter(y_test, pred, alpha=0.3, s=10, color='steelblue')
    lim = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lim, lim, 'r--', lw=2, label='Perfect prediction')
    ax.set_xlabel('Actual', fontweight='bold')
    ax.set_ylabel('Predicted', fontweight='bold')
    ax.set_title(f'{name}\nRMSE={root_mean_squared_error(y_test, pred):.1f} | R²={r2_score(y_test, pred):.3f}',
                 fontweight='bold')
    ax.legend(fontsize=9)
plt.suptitle('Actual vs. Predicted — Test Set', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 8. Limitations & Future Work

**Known limitations:**
- Simple average ensemble assumes equal model quality — a weighted or stacked ensemble could be better
- `high_event_day` flag is derived from the data itself (p95 of casual); in production this would require a known event calendar
- No explicit handling of temporal autocorrelation — LSTM or temporal convolutions could capture lag effects
- Model trained on 2011–2012 data; distribution shift to current conditions would require retraining

**Potential improvements:**
- Stacked generalization (train a meta-learner on NN + XGBoost out-of-fold predictions)
- Add lag features: `cnt_lag_1h`, `cnt_lag_24h`, `cnt_rolling_mean_7d`
- Integrate weather forecast data for future-hour prediction
- Deploy as an API for real-time hourly prediction
